In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# import torch

# from transformers import (
#     AutoTokenizer,
#     AutoModelForMultipleChoice,
#     Trainer,
#     TrainingArguments
# )

# from datasets import Dataset

# from peft import (
#     LoraConfig,
#     get_peft_model,
#     TaskType
# )

In [3]:
# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# train.head()

In [4]:
# # q1
# label_map = {
#     "A":0,
#     "B":1,
#     "C":2,
#     "D":3,
#     "E":4
# }

# train["label"] = train["answer"].map(label_map)

# print(train.loc[150,"label"])

In [5]:
# #q2
# row = train.iloc[0]

# text = str(row["prompt"]) + " [SEP] " + str(row["B"])

# print(text)

# print(len(text))

In [6]:
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [7]:
# # q3
# row = train.iloc[0]

# choices = [
#     str(row["prompt"])+" [SEP] "+str(row["A"]),
#     str(row["prompt"])+" [SEP] "+str(row["B"]),
#     str(row["prompt"])+" [SEP] "+str(row["C"]),
#     str(row["prompt"])+" [SEP] "+str(row["D"]),
#     str(row["prompt"])+" [SEP] "+str(row["E"])
# ]

# enc = tokenizer(
#     choices,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# input_ids = enc["input_ids"].unsqueeze(0)

# print(input_ids.shape)

In [8]:
# # q4
# all_input_ids = []

# for i in range(16):

#     row = train.iloc[i]

#     choices = [
#         row["prompt"]+" [SEP] "+row["A"],
#         row["prompt"]+" [SEP] "+row["B"],
#         row["prompt"]+" [SEP] "+row["C"],
#         row["prompt"]+" [SEP] "+row["D"],
#         row["prompt"]+" [SEP] "+row["E"]
#     ]

#     enc = tokenizer(
#         choices,
#         padding="max_length",
#         truncation=True,
#         max_length=128,
#         return_tensors="pt"
#     )

#     all_input_ids.append(enc["input_ids"])

# batch = torch.stack(all_input_ids)

# print(batch.shape)

# print(batch.numel())

In [9]:
# # q5
# model = AutoModelForMultipleChoice.from_pretrained(
#     "bert-base-uncased"
# )

# outputs = model(
#     input_ids=input_ids,
#     attention_mask=enc["attention_mask"].unsqueeze(0)
# )

# print(outputs.logits.shape)

In [10]:
# # q6
# label = torch.tensor([train.loc[0,"label"]])

# outputs = model(
#     input_ids=input_ids,
#     attention_mask=enc["attention_mask"].unsqueeze(0),
#     labels=label
# )

# print(outputs.loss)

# print(outputs.loss.shape)

# print(outputs.loss.dim())

In [11]:
# # q7
# from transformers import AutoModelForMultipleChoice
# from peft import LoraConfig, get_peft_model, TaskType

# model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=["query","value"],
#     lora_dropout=0.1,
#     bias="none",
#     task_type=TaskType.SEQ_CLS
# )

# model = get_peft_model(model, config)

# model.print_trainable_parameters()

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# print(trainable)

In [12]:
# # q8
# small = train.iloc[:100].copy()

# data = []

# for _,row in small.iterrows():

#     choices = [
#         row["prompt"]+" [SEP] "+row["A"],
#         row["prompt"]+" [SEP] "+row["B"],
#         row["prompt"]+" [SEP] "+row["C"],
#         row["prompt"]+" [SEP] "+row["D"],
#         row["prompt"]+" [SEP] "+row["E"]
#     ]

#     enc = tokenizer(
#         choices,
#         padding="max_length",
#         truncation=True,
#         max_length=128
#     )

#     data.append({
#         "input_ids":enc["input_ids"],
#         "attention_mask":enc["attention_mask"],
#         "labels":row["label"]
#     })

# dataset = Dataset.from_list(data)

# print(len(dataset))

# print(torch.tensor(dataset[0]["input_ids"]).shape)

In [13]:
# # q9
# tiny = train.iloc[:32].copy()

# data=[]

# for _,row in tiny.iterrows():

#     choices=[
#         row["prompt"]+" [SEP] "+row["A"],
#         row["prompt"]+" [SEP] "+row["B"],
#         row["prompt"]+" [SEP] "+row["C"],
#         row["prompt"]+" [SEP] "+row["D"],
#         row["prompt"]+" [SEP] "+row["E"]
#     ]

#     enc=tokenizer(
#         choices,
#         padding="max_length",
#         truncation=True,
#         max_length=64
#     )

#     data.append({
#         "input_ids":enc["input_ids"],
#         "attention_mask":enc["attention_mask"],
#         "labels":row["label"]
#     })

# dataset=Dataset.from_list(data)

In [14]:
# args=TrainingArguments(

#     output_dir="out",

#     per_device_train_batch_size=4,

#     gradient_accumulation_steps=1,

#     max_steps=4,

#     logging_steps=1,

#     report_to="none"

# )

In [15]:
# trainer=Trainer(
#     model=model,
#     args=args,
#     train_dataset=dataset
# )

# trainer.train()

# print(trainer.state.global_step)

In [16]:
# # q10
#row = train.iloc[0]

# choices = [
#     row["prompt"] + " [SEP] " + row["A"],
#     row["prompt"] + " [SEP] " + row["B"],
#     row["prompt"] + " [SEP] " + row["C"],
#     row["prompt"] + " [SEP] " + row["D"],
#     row["prompt"] + " [SEP] " + row["E"],
# ]

# enc = tokenizer(
#     choices,
#     padding="max_length",
#     truncation=True,
#     max_length=64,
#     return_tensors="pt"
# )

# device = next(model.parameters()).device

# model.eval()

# with torch.no_grad():
#     outputs = model(
#         input_ids=enc["input_ids"].unsqueeze(0).to(device),
#         attention_mask=enc["attention_mask"].unsqueeze(0).to(device)
#     )

# probs = torch.softmax(outputs.logits, dim=-1)

# print(probs)

# print(round(probs[0][4].item(), 4))